# 15-01 从像素比较到动作条件表征预测

本实验对应第 15 章，分三步检查：像素误差是否符合控制目标；固定视觉表征怎样形成“当前表征与动作 → 未来表征”的训练样本；丢失位置信息的表征为什么即使容易预测也不能支持选动作。

实验只依赖 NumPy、Matplotlib 和 PyTorch，CPU 即可。手工位置特征充当可读的固定编码器，用来展示 JEPA 式数据流；它不是 DINO-WM、LeWM 或任何完整 JEPA 的复现。


## 1. 同一个目标，两种不同的错误

目标是把白色方块放到左侧。候选 A 位置正确但背景变亮；候选 B 背景相同但位置移到右侧。先肉眼判断，再计算误差。

In [ ]:
import numpy as np  # 生成可控的教学图像
import matplotlib.pyplot as plt  # 并排显示目标和候选
def scene(x, background):  # 根据中心与背景亮度创建图像
    image = np.full((32, 32), background, dtype=float)  # 创建指定亮度的背景
    image[12:20, x-4:x+4] = 1.0  # 放入固定大小的白色方块
    return image  # 返回单通道图像
goal = scene(10, 0.0)  # 指定目标位置与暗背景
correct_position = scene(10, 0.4)  # 保持位置正确但改变背景
wrong_position = scene(24, 0.0)  # 保持背景正确但改变位置
images = [goal, correct_position, wrong_position]  # 按解释顺序收集图像
titles = ["Goal", "A: right position", "B: wrong position"]  # 用简洁标题说明对照关系
fig, axes = plt.subplots(1, 3, figsize=(9, 3))  # 创建三幅图的画布
for ax, image, title in zip(axes, images, titles):  # 依次显示目标和候选
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)  # 固定亮度范围以公平比较背景
    ax.set_title(title)  # 标明图像对应的候选
    ax.axis("off")  # 隐藏与判断无关的刻度
plt.show()  # 展示视觉结果


### 先读图，不先看分数

左图是目标；中图 A 的方块位置正确，只是大面积背景变亮；右图 B 的方块明显偏离目标。任务只关心位置，所以应偏好 A。固定亮度范围很重要，自动拉伸颜色可能把背景变化隐藏起来。

## 2. 比较两种误差究竟在奖励什么

像素均方误差比较所有像素。手工位置特征只取亮度高于 0.8 的区域中心；这依赖已知颜色规则，不代表真实图像总能如此处理。

In [ ]:
def position_feature(image):  # 提取仅用于教学的方块中心
    ys, xs = np.nonzero(image > 0.8)  # 按阈值找到白色方块
    return np.array([xs.mean(), ys.mean()])  # 返回横向与纵向中心
pixel_errors = [np.mean((candidate - goal) ** 2) for candidate in images[1:]]  # 计算整张图的像素误差
position_errors = [np.linalg.norm(position_feature(candidate) - position_feature(goal)) for candidate in images[1:]]  # 计算位置距离
fig, axes = plt.subplots(1, 2, figsize=(8, 3))  # 分开显示不同单位的误差
axes[0].bar(["A", "B"], pixel_errors, color=["#497f99", "#b67d4c"])  # 左图比较像素误差
axes[0].set(ylabel="Pixel MSE", title="All pixels count")  # 标明亮度误差含义
axes[1].bar(["A", "B"], position_errors, color=["#497f99", "#b67d4c"])  # 右图比较中心距离
axes[1].set(ylabel="Center distance (pixels)", title="Position matters")  # 标明位置误差单位
fig.tight_layout()  # 留出标题与坐标标签空间
plt.show()  # 显示不同评价规则给出的排名
assert pixel_errors[0] > pixel_errors[1]  # 核对背景变化支配了本例的像素比较
assert position_errors[0] == 0 and position_errors[1] > 0  # 核对位置特征正确区分了候选


### 两张图为什么给出相反判断

误差越小越好。左图偏好 B，因为大量背景像素没有变化；右图偏好 A，因为它只比较中心位置。不能比较左右柱子的绝对高度：一个是亮度平方误差，一个是像素距离，单位不同。

这不说明像素训练必然失败，只说明评价指标可能与控制目标不一致。真实模型还需要方向、形状和接触等信息。

## 3. 固定编码器以后，怎样学习动作条件预测

前面的位置特征可以看作一个固定编码器：输入图像，输出方块中心坐标。现在生成许多“当前位置、水平动作、未来位置”的样本，训练一个小型预测器。预测器只能看到当前特征和动作；未来图像经过同一个固定编码器得到训练答案。

这个实验特意使用简单平移，使输入、动作和答案都能直接核对。它展示的是动作条件表征预测的数据关系，不代表真实视觉编码器只需要两个数。


In [ ]:
import torch  # 训练最小的动作条件表征预测器
from torch import nn  # 使用线性层与非线性激活函数
rng = np.random.default_rng(15)  # 固定训练样本以便重复核对
torch.manual_seed(15)  # 固定网络初始化与优化过程
sample_count = 800  # 设置教学训练样本数量
current_x = rng.integers(5, 27, size=sample_count)  # 随机生成当前方块横向位置
horizontal_action = rng.integers(-4, 5, size=sample_count)  # 随机生成向左或向右的像素位移
future_x = np.clip(current_x + horizontal_action, 4, 27)  # 根据动作得到真实未来横向位置
current_feature = np.column_stack((current_x / 31.0, np.full(sample_count, 15.5 / 31.0)))  # 用固定编码器的结果表示当前图像
future_feature = np.column_stack((future_x / 31.0, np.full(sample_count, 15.5 / 31.0)))  # 用真实未来图像形成表征答案
predictor_input = np.column_stack((current_feature, horizontal_action / 4.0)).astype(np.float32)  # 拼接当前表征和归一化动作
predictor_target = future_feature.astype(np.float32)  # 保存未来表征作为监督答案
feature_predictor = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 2))  # 建立三输入两输出的表征预测器
optimizer = torch.optim.Adam(feature_predictor.parameters(), lr=0.02)  # 设置参数更新方法
inputs_tensor = torch.tensor(predictor_input)  # 将全部训练输入转成张量
targets_tensor = torch.tensor(predictor_target)  # 将全部未来表征答案转成张量
loss_history = []  # 保存每轮未来表征预测误差
for update in range(300):  # 重复训练三百轮
    predicted_feature = feature_predictor(inputs_tensor)  # 根据当前特征和动作预测未来特征
    loss = ((predicted_feature - targets_tensor) ** 2).mean()  # 计算预测与真实未来特征的均方误差
    optimizer.zero_grad()  # 清除上一轮梯度
    loss.backward()  # 计算误差对预测器参数的影响
    optimizer.step()  # 更新预测器而不更新固定编码器
    loss_history.append(float(loss.detach()))  # 保存本轮实际误差
probe_actions = np.arange(-4, 5)  # 为同一当前位置准备九种不同动作
probe_current = 16  # 固定用于检查动作条件的当前横向位置
probe_input = np.column_stack((np.full(9, probe_current / 31.0), np.full(9, 15.5 / 31.0), probe_actions / 4.0)).astype(np.float32)  # 组合九条预测输入
with torch.no_grad():  # 评测时关闭梯度记录
    probe_prediction = feature_predictor(torch.tensor(probe_input)).numpy()[:, 0] * 31.0  # 将未来横向特征恢复为像素位置
probe_truth = np.clip(probe_current + probe_actions, 4, 27)  # 计算同一组动作的真实未来位置
print(f'一条训练输入：[当前 x 特征={predictor_input[0, 0]:.3f}, 当前 y 特征={predictor_input[0, 1]:.3f}, 动作={predictor_input[0, 2]:+.3f}]')  # 展示预测器真正看到的三个数
print(f'对应未来表征答案：{predictor_target[0].round(3)}')  # 展示真实未来图像编码出的两个答案
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))  # 并排显示训练误差与动作条件预测
axes[0].plot(loss_history, color='#497f99')  # 绘制未来表征预测误差
axes[0].set(xlabel='training update', ylabel='feature MSE', yscale='log', title='Predict future feature')  # 标明误差单位与训练过程
axes[1].plot(probe_actions, probe_truth, 'o-', color='black', label='true future x')  # 绘制不同动作的真实未来位置
axes[1].plot(probe_actions, probe_prediction, 's--', color='#b67d4c', label='predicted future x')  # 绘制网络对相同动作的预测
axes[1].set(xlabel='horizontal action (pixels)', ylabel='future x (pixels)', title='Same image, different actions')  # 标明动作条件测试含义
axes[1].legend()  # 区分真实值与网络预测
fig.tight_layout()  # 避免两个子图的标签重叠
plt.show()  # 输出训练与预测对照
assert np.mean(np.abs(probe_prediction - probe_truth)) < 0.8  # 检查最小预测器已学会明显的动作影响


### 这两幅图对应正文中的哪条数据流

左图的下降表示预测器逐渐接近真实未来表征。这里固定编码器是手工位置提取函数，所以训练只更新 `feature_predictor`；未来图像只负责产生答案，不作为预测器输入。

右图固定同一个当前图像，仅改变动作。黑线是动作真正造成的未来横向位置，虚线是网络预测。两条线随动作一起变化，说明预测器确实读取了动作。若所有动作都得到同一输出，即使训练平均误差不大，也不能用于第 16 章比较候选动作。

这个结果只证明一个低维平移例子的数据关系正确。真实图像块特征包含更多维度，接触、遮挡与旋转也不会像这里一样完全线性。


## 4. 表征更小，就一定更好吗

最后故意构造一个失败编码器：它只取整张图的最大亮度。目标、候选 A 和候选 B 都含白色方块，因此三个结果都是 1。这样的表征几乎不需要学习就能“预测准确”，却无法指出方块在哪里，也不能判断哪个候选到达目标。


In [ ]:
collapsed = [float(image.max()) for image in images]  # 用单一亮度模拟丢失位置信息的表征
print("目标、候选 A、候选 B 的单值表征：", collapsed)  # 显示不同场景不可区分
print("三个场景的位置特征：", [position_feature(image).tolist() for image in images])  # 与保留位置信息的表示对照
assert len(set(collapsed)) == 1  # 核对单值表征不能区分图像


### 为什么“预测误差为零”仍然可能失败

单值表征把位置不同的三张图都变成了 1，因此无论输入什么动作，预测器只要继续输出 1 就能得到零误差。这正是表征坍缩或信息丢失为什么危险：误差很小，但控制所需的区别已经不存在。

对照上一个实验，二维位置特征会随动作和位置变化，所以可以检查未来是否接近目标。本 Notebook 没有实现 LeWM 的 SIGReg；它只用一个完全可见的失败例子说明，评价视觉世界模型时必须同时问“未来特征是否预测准确”和“这些特征是否保留任务需要的信息”。
